# Train YOLOv3 for FPGA AI Suite (Colab T4 GPU → Agilex 7 F-series)

Trains YOLOv3 (transfer learning) on a T4 **GPU**, then exports an **FPGA-friendly OpenVINO IR**: the raw convolutional detection heads only (conv / BN / LeakyReLU / upsample / concat — the operators the DLA maps cleanly). YOLO **decode + NMS run on the host**, not on the FPGA (see `yolov3/yolo_postprocess.py`).

**Before you start:**
1. `Runtime → Change runtime type → **T4 GPU**`.
2. Zip the dataset on your computer and upload the single zip to Google Drive (`MyDrive`):
   ```bash
   cd ~/Downloads/PCB_yolov3/datasets
   zip -r -q ../unified_pku_yolo_gray640.zip unified_pku_yolo_gray640
   ```

Every script cell uses an absolute `REPO` path, so it survives runtime reconnects.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Install OpenVINO + NNCF + deps

In [ ]:
!pip install -q "openvino>=2025.0" nncf opencv-python-headless tensorboard
import openvino as ov, nncf; print('OpenVINO', ov.__version__, '| NNCF', nncf.__version__)

## 3. Get the training code. `REPO` is an absolute path used everywhere below.

In [ ]:
import os
REPO = '/content/PCB_yolov3'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/EasonLi292/PCB_yolov3.git {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!ls {REPO}/yolov3

## 4. Load your data from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_NAME = 'unified_pku_yolo_gray640'
ZIP_ON_DRIVE = f'/content/drive/MyDrive/{DATASET_NAME}.zip'
!cp "{ZIP_ON_DRIVE}" /content/
!mkdir -p /content/data && unzip -q -o /content/{DATASET_NAME}.zip -d /content/data
DATA_DIR = f'/content/data/{DATASET_NAME}'
RUN_DIR = f'{REPO}/runs/{DATASET_NAME}'
print('DATA_DIR =', DATA_DIR)
import pathlib
for s in ['train', 'val', 'test']:
    d = pathlib.Path(DATA_DIR) / s / 'images'
    print(f'  {s}: {len(list(d.glob("*"))) if d.is_dir() else 0} images')
assert (pathlib.Path(DATA_DIR) / 'data.yaml').exists(), 'data.yaml not found — check the zip structure'

## 5. Download the pretrained Darknet weights (237 MB)

In [ ]:
!cd {REPO} && mkdir -p weights && ([ -f weights/yolov3.weights ] || wget -q --show-progress https://pjreddie.com/media/files/yolov3.weights -O weights/yolov3.weights)
!ls -lh {REPO}/weights/yolov3.weights

## 6. Train — phase 1 (frozen backbone)
Trains the detection heads on the frozen Darknet-53 features, starting fresh from the COCO backbone (`RESUME_FROM = ''`). Uses **k-means anchors** (`yolov3/anchors.json`, loaded automatically) and **online augmentation** (flips, 90° rotation, brightness). To continue from a Drive checkpoint instead, set `RESUME_FROM` to its path.

The best checkpoint is saved to `yolov3_best.weights.h5` **every improving epoch** (interrupt-safe), and the FPGA export reads it directly. On a T4, `--batch 8` fits; on an H100 use `--batch 32` and bump epochs.

In [ ]:
# Phase 1: fresh transfer from the COCO backbone (RESUME_FROM='').
# To instead continue from a checkpoint on Drive, set RESUME_FROM to its path.
import os
RESUME_FROM = ''
EPOCHS_P1 = 30

if RESUME_FROM and os.path.exists(RESUME_FROM):
    os.makedirs(RUN_DIR, exist_ok=True)
    !cp "{RESUME_FROM}" "{RUN_DIR}/resume_start.weights.h5"
    print('Phase 1: resuming (frozen) from', RESUME_FROM)
    !cd {REPO} && python yolov3/train_yolov3.py --data "{DATA_DIR}" --resume runs/{DATASET_NAME}/resume_start.weights.h5 --epochs {EPOCHS_P1} --batch 8
else:
    print('Phase 1: fresh transfer from the COCO backbone (k-means anchors, augmentation ON)')
    !cd {REPO} && python yolov3/train_yolov3.py --data "{DATA_DIR}" --weights weights/yolov3.weights --epochs {EPOCHS_P1} --batch 8

## 7. (optional) Train — phase 2: unfreeze + fine-tune
Resumes from phase-1's best and fine-tunes the whole network at a low LR. Usually the biggest accuracy gain, and a better-trained model also **quantizes more robustly**. This does NOT change the graph, so it has zero effect on FPGA compatibility.

In [ ]:
!cd {REPO} && python yolov3/train_yolov3.py --data "{DATA_DIR}" \
    --resume runs/{DATASET_NAME}/yolov3_best.weights.h5 --unfreeze --lr 1e-4 --epochs 15 --batch 8

## 8. Export FPGA IR (raw conv heads, FP32)
Builds the OpenVINO IR with **3 raw detection-head outputs and no decode in the graph** (clean for the Agilex DLA). Reads `yolov3_best.weights.h5`, so it works even if you stopped training early. **This FP32 IR is what you feed to the AI Suite compiler** — let `dla_compiler` do the INT8 calibration (its DLA-tuned flow; see the INT8 note in the last cell).

In [ ]:
!cd {REPO} && python yolov3/export_fpga.py \
    --weights runs/{DATASET_NAME}/yolov3_best.weights.h5 \
    --out runs/{DATASET_NAME}/openvino_fpga --nc 6
open(f'{RUN_DIR}/openvino_fpga/classes.txt','w').write(
    '\n'.join(['missing_hole','mouse_bite','open_circuit','short','spur','spurious_copper'])+'\n')
!ls -lh {RUN_DIR}/openvino_fpga

## 9. Bundle and download the FPGA IR
Downloads `yolov3_fpga_fp32.{xml,bin}` + `classes.txt` and backs up to Drive (plus the trained weights). Drop into `runs/unified_pku_yolo_gray640/openvino_fpga/` on your computer; I'll evaluate it locally with host-side decode.

In [ ]:
import shutil
shutil.make_archive('/content/yolov3_fpga_ir', 'zip', f'{RUN_DIR}/openvino_fpga')
!cp /content/yolov3_fpga_ir.zip /content/drive/MyDrive/
!cp {RUN_DIR}/yolov3_best.weights.h5 /content/drive/MyDrive/{DATASET_NAME}_yolov3_best.weights.h5
from google.colab import files
files.download('/content/yolov3_fpga_ir.zip')

## FPGA AI Suite (Agilex 7 F-series) — next steps on your side

1. **Feed `yolov3_fpga_fp32.xml` to the AI Suite compiler** for your Agilex 7 `.arch`. The graph is one clean convolutional subgraph (3 raw outputs), so it should partition fully onto the DLA.
2. **Decode + NMS run on the host** — port `yolov3/yolo_postprocess.py` (numpy, no TF) next to your FPGA runtime. It turns the 3 raw output tensors into boxes. Anchors live there as constants, so you can swap in k-means anchors later with **zero FPGA impact**.
3. **INT8:** prefer the AI Suite compiler's own INT8 calibration (run on the FP32 IR with a calibration image set) — it's tuned for the DLA. We tested NNCF post-training INT8 here and it degraded accuracy badly on the early/undertrained weights, so treat the optional NNCF path as experimental and **always re-check mAP** with `analyze_openvino.py` before trusting it. A fully-trained model (phase 1 + phase 2) quantizes far better.
4. **Verify operators against your `.arch`:** LeakyReLU and upsample/resize are the two to confirm are enabled in your DLA architecture config.

### (optional, experimental) NNCF INT8 — verify before use
```
!cd {REPO} && python yolov3/export_fpga.py --weights runs/{DATASET_NAME}/yolov3_best.weights.h5 \
    --out runs/{DATASET_NAME}/openvino_fpga --nc 6 --int8 --calib-data "{DATA_DIR}/train" --calib-n 300
```